In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [42]:
medals = pd.read_csv("../data/processed/olympic_1896_2024_extended.csv")
medals.head()

,Name,Sex,NOC,Year,Sport,Event,Medal
0,A Dijiang,M,CHN,1992,Basketball,Basketball Men's Basketball,NaN
1,A Lamusi,M,CHN,2012,Judo,Judo Men's Extra-Lightweight,NaN
2,Gunnar Nielsen Aaby,M,DEN,1920,Football,Football Men's Football,NaN
3,Edgar Lindenau Aabye,M,DEN,1900,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,Christine Jacoba Aaftink,F,NED,1988,Speed Skating,Speed Skating Women's 500 metres,NaN


In [43]:
# Keep only Summer Olympics
summer_years = [
1896,1900,1904,1908,1912,
1920,1924,1928,1932,1936,
1948,1952,1956,1960,1964,
1968,1972,1976,1980,1984,
1988,1992,1996,2000,2004,
2008,2012,2016,2020,2024
]

summer = medals[medals["Year"].isin(summer_years)]

india = summer[summer["NOC"] == "IND"].copy()

india.head()

,Name,Sex,NOC,Year,Sport,Event,Medal
505,S. Abdul Hamid,M,IND,1928,Athletics,Athletics Men's 110 metres Hurdles,NaN
506,S. Abdul Hamid,M,IND,1928,Athletics,Athletics Men's 400 metres Hurdles,NaN
895,Shiny Kurisingal Abraham-Wilson,F,IND,1984,Athletics,Athletics Women's 800 metres,NaN
896,Shiny Kurisingal Abraham-Wilson,F,IND,1984,Athletics,Athletics Women's 4 x 400 metres Relay,NaN
897,Shiny Kurisingal Abraham-Wilson,F,IND,1988,Athletics,Athletics Women's 800 metres,NaN


In [44]:
india_sport_year = (
    india.drop_duplicates(subset=["Year", "Sport", "Event", "Medal"])
    .groupby(["Year", "Sport"])
    .size()
    .reset_index(name="Total")
    .sort_values(["Sport", "Year"])
)

india_sport_year.head()

,Year,Sport,Total
50,1964,Alpine Skiing,1
59,1968,Alpine Skiing,3
92,1988,Alpine Skiing,2
104,1992,Alpine Skiing,2
3,1924,Alpinism,1


In [45]:
# Career average WITHOUT leakage
india_sport_year["career_avg"] = (
    india_sport_year
        .groupby("Sport")["Total"]
        .expanding()
        .mean()
        .shift(1)                 # <-- prevents leakage
        .reset_index(level=0, drop=True)
)

# Momentum
india_sport_year["delta_last"] = (
    india_sport_year
        .groupby("Sport")["Total"]
        .diff()
)

# Fill initial NaNs
india_sport_year[["career_avg", "delta_last"]] = (
    india_sport_year[["career_avg", "delta_last"]]
    .fillna(0)
)

india_sport_year.head(10)

,Year,Sport,Total,career_avg,delta_last
50,1964,Alpine Skiing,1,0.0,0.0
59,1968,Alpine Skiing,3,1.0,2.0
92,1988,Alpine Skiing,2,2.0,-1.0
104,1992,Alpine Skiing,2,2.0,0.0
3,1924,Alpinism,1,2.0,0.0
93,1988,Archery,2,1.0,0.0
105,1992,Archery,2,2.0,0.0
117,1996,Archery,2,2.0,0.0
143,2004,Archery,4,2.0,2.0
157,2008,Archery,3,2.5,-1.0


In [46]:
# Train on history up to 2020
train = india_sport_year[india_sport_year["Year"] <= 2020]

# Test on 2024
test = india_sport_year[india_sport_year["Year"] == 2024]

features = ["career_avg", "delta_last"]
target = "Total"

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

In [47]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model = XGBRegressor(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate
pred_test = model.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)
r2 = r2_score(y_test, pred_test)

print("Layer 2 Test MAE:", round(mae, 3))
print("Layer 2 Test R2 :", round(r2, 3))

Layer 2 Test MAE: 2.552
Layer 2 Test R2 : -15.139


In [48]:
# Get latest year per sport (2024 rows)
latest = india_sport_year[india_sport_year["Year"] == 2024].copy()

# Predict 2028 using 2024 career_avg & delta_last
latest["Pred_2028"] = model.predict(latest[features])

# Clean negatives
latest["Pred_2028"] = latest["Pred_2028"].clip(lower=0)

# Round for interpretation
latest["Pred_2028"] = latest["Pred_2028"].round(2)

latest[["Sport", "Total", "career_avg", "delta_last", "Pred_2028"]]\
    .sort_values("Pred_2028", ascending=False)

,Sport,Total,career_avg,delta_last,Pred_2028
203,Athletics,1,7.680000,0.0,7.36
206,Wrestling,1,5.300000,-1.0,3.60
205,Shooting,3,4.294118,-8.0,3.15
204,Hockey,1,1.095238,0.0,2.10


In [49]:
print("Total Predicted Medals 2028 (Layer 2):",
      round(latest["Pred_2028"].sum(), 2))

Total Predicted Medals 2028 (Layer 2): 16.21


In [50]:
# Get all sports India ever participated in
all_sports = india["Sport"].unique()

# Get all summer years
all_years = sorted(india["Year"].unique())

# Create full grid
full_grid = pd.MultiIndex.from_product(
    [all_years, all_sports],
    names=["Year", "Sport"]
).to_frame(index=False)

# Merge with actual medal counts
india_sport_year_full = full_grid.merge(
    india_sport_year,
    on=["Year", "Sport"],
    how="left"
)

# Fill missing medals with 0
india_sport_year_full["Total"] = india_sport_year_full["Total"].fillna(0)

india_sport_year_full = india_sport_year_full.sort_values(["Sport", "Year"])

india_sport_year_full.head()

,Year,Sport,Total,career_avg,delta_last
20,1900,Alpine Skiing,0.0,NaN,NaN
45,1920,Alpine Skiing,0.0,NaN,NaN
70,1924,Alpine Skiing,0.0,NaN,NaN
95,1928,Alpine Skiing,0.0,NaN,NaN
120,1932,Alpine Skiing,0.0,NaN,NaN


In [51]:
india_sport_year_full["career_avg"] = (
    india_sport_year_full
        .groupby("Sport")["Total"]
        .expanding()
        .mean()
        .shift(1)
        .reset_index(level=0, drop=True)
)

india_sport_year_full["delta_last"] = (
    india_sport_year_full
        .groupby("Sport")["Total"]
        .diff()
)

india_sport_year_full[["career_avg", "delta_last"]] = (
    india_sport_year_full[["career_avg", "delta_last"]]
    .fillna(0)
)

In [52]:
india_sport_year_full["medal_binary"] = (
    india_sport_year_full["Total"] > 0
).astype(int)

train = india_sport_year_full[india_sport_year_full["Year"] <= 2020]
test  = india_sport_year_full[india_sport_year_full["Year"] == 2024]

features = ["career_avg", "delta_last"]

X_train = train[features]
y_train = train["medal_binary"]

X_test = test[features]
y_test = test["medal_binary"]

In [53]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

clf = XGBClassifier(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

clf.fit(X_train, y_train)

probs = clf.predict_proba(X_test)[:,1]

print("ROC-AUC:", roc_auc_score(y_test, probs))

ROC-AUC: 0.9166666666666666


In [54]:
latest = india_sport_year_full[
    india_sport_year_full["Year"] == 2024
].copy()

latest["Medal_Prob_2028"] = clf.predict_proba(
    latest[features]
)[:,1]

latest[["Sport", "Medal_Prob_2028"]]\
    .sort_values("Medal_Prob_2028", ascending=False)

,Sport,Medal_Prob_2028
640,Shooting,0.995084
635,Wrestling,0.946206
625,Athletics,0.765097
637,Badminton,0.742895
642,Boxing,0.718711
627,Weightlifting,0.690308
626,Table Tennis,0.455576
634,Cycling,0.447851
628,Judo,0.447851
639,Archery,0.368178


In [55]:
# Train only on years where medal was won
train_positive = train[
    (train["Total"] > 0) &
    (train["Year"] >= 2000)
]

X_train_pos = train_positive[features]
y_train_pos = train_positive["Total"]

In [56]:
from xgboost import XGBRegressor

count_model = XGBRegressor(
    n_estimators=120,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

count_model.fit(X_train_pos, y_train_pos)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=120,
             n_jobs=None, num_parallel_tree=None, ...)

In [57]:
latest["Count_if_Medal_2028"] = count_model.predict(latest[features])
latest["Count_if_Medal_2028"] = latest["Count_if_Medal_2028"].clip(lower=0)

In [58]:
latest["Expected_Medals_2028"] = (
    latest["Medal_Prob_2028"] *
    latest["Count_if_Medal_2028"]
)

latest["Expected_Medals_2028"] = latest["Expected_Medals_2028"].round(3)

latest_sorted = latest[
    ["Sport", "Medal_Prob_2028", 
     "Count_if_Medal_2028", 
     "Expected_Medals_2028"]
].sort_values("Expected_Medals_2028", ascending=False)

latest_sorted.head(15)

,Sport,Medal_Prob_2028,Count_if_Medal_2028,Expected_Medals_2028
640,Shooting,0.995084,9.310010,9.264
625,Athletics,0.765097,8.328181,6.372
642,Boxing,0.718711,5.059172,3.636
635,Wrestling,0.946206,3.312075,3.134
627,Weightlifting,0.690308,2.090020,1.443
637,Badminton,0.742895,1.688281,1.254
626,Table Tennis,0.455576,2.220702,1.012
634,Cycling,0.447851,1.991912,0.892
628,Judo,0.447851,1.991912,0.892
639,Archery,0.368178,2.220710,0.818


In [59]:
print(
    "Total Expected Medals 2028 (Layer 2):",
    round(latest["Expected_Medals_2028"].sum(), 2)
)

Total Expected Medals 2028 (Layer 2): 32.23
